<a href="https://colab.research.google.com/github/GIRIAYUSH/playing-with-anns/blob/main/notebooks/M3_loss_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M3 · Loss Functions

> *"You can't improve what you can't measure."*

A loss function takes your model's predictions and the ground truth,
and collapses everything into **one number** — how wrong are you?

The shape of that number, and how it changes with your predictions,
is what the optimizer will navigate later.

**What we'll cover:**

| Loss Function | When to use |
|---|---|
| Binary Cross-Entropy (BCE) | 2 classes — yes/no |
| Cross-Entropy | 3+ mutually exclusive classes |
| Categorical Cross-Entropy | Same as CE, different input format |
| Negative Log-Likelihood (NLL) | When you handle softmax yourself |
| Label Smoothing CE | When you want to avoid overconfident predictions |
| Focal Loss | When your classes are heavily imbalanced |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

torch.manual_seed(42)

## Intuition First — What Makes a Good Loss?

Before diving in, let's build intuition for what a loss function *should* do.

Imagine your model outputs a probability for each class.  
A good loss function should:

1. Return **near zero** when predictions are confident and correct
2. Return a **large number** when predictions are confident and wrong
3. Be **differentiable** so gradients can flow back

Let's visualise this before writing a single loss function.

In [ ]:
p = np.linspace(0.001, 0.999, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("What a Good Loss Function Looks Like", fontsize=13, fontweight='bold')

# --- Left: -log(p) for correct class ---
axes[0].plot(p, -np.log(p), color='steelblue', linewidth=2.5)
axes[0].fill_between(p, -np.log(p), alpha=0.1, color='steelblue')
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='50% confidence')
axes[0].scatter([0.9], [-np.log(0.9)], color='green', s=100, zorder=5, label='Confident & correct → low loss')
axes[0].scatter([0.1], [-np.log(0.1)], color='red',   s=100, zorder=5, label='Confident & wrong → high loss')
axes[0].set_title("Loss vs Predicted Probability (True class)")
axes[0].set_xlabel("Predicted probability for correct class")
axes[0].set_ylabel("Loss = -log(p)")
axes[0].set_ylim(0, 5)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# --- Right: what "confident and wrong" looks like ---
scenarios = {
    'p=0.9 (confident, correct)': 0.9,
    'p=0.7 (fairly sure, correct)': 0.7,
    'p=0.5 (uncertain)': 0.5,
    'p=0.3 (leaning wrong)': 0.3,
    'p=0.1 (confident, WRONG)': 0.1,
}
colors = ['green', 'limegreen', 'orange', 'tomato', 'red']
losses = [-np.log(v) for v in scenarios.values()]
bars = axes[1].barh(list(scenarios.keys()), losses, color=colors, alpha=0.8)
axes[1].set_xlabel("Loss value")
axes[1].set_title("Loss for Different Prediction Confidences\n(True label = class 1)")
axes[1].axvline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(alpha=0.3, axis='x')
for bar, loss in zip(bars, losses):
    axes[1].text(loss + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{loss:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## A Quick Note on Information Theory

Cross-Entropy comes from **information theory**, not just statistics.

**Entropy** measures uncertainty in a distribution:
$$H(p) = -\sum_i p_i \log p_i$$

**Cross-Entropy** measures how well distribution $q$ (your predictions)
approximates the true distribution $p$ (the labels):
$$H(p, q) = -\sum_i p_i \log q_i$$

When $p$ is a one-hot label (one class is 100% true), this simplifies to:
$$\mathcal{L} = -\log(\hat{y}_{true\_class})$$

This is the core formula behind **all** cross-entropy losses.  
We're simply asking: *"What probability did you assign to the correct class?"*  
Then we take `-log` of it to penalise low confidence.

In [ ]:
p = np.linspace(0.001, 0.999, 300)
entropy = -(p * np.log(p) + (1-p) * np.log(1-p))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Entropy curve ---
axes[0].plot(p, entropy, color='mediumpurple', linewidth=2.5)
axes[0].fill_between(p, entropy, alpha=0.15, color='mediumpurple')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Max uncertainty at p=0.5')
axes[0].scatter([0.0, 1.0], [0, 0], color='green', s=80, zorder=5, label='Zero uncertainty (certain)')
axes[0].set_title("Binary Entropy H(p)\nHighest when most uncertain")
axes[0].set_xlabel("Probability p")
axes[0].set_ylabel("Entropy")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# --- Cross entropy vs entropy ---
axes[1].plot(p, entropy, color='mediumpurple', linewidth=2, label='Entropy H(p,p) — perfect model')
axes[1].plot(p, -(p * np.log(0.6) + (1-p) * np.log(0.4)),
             color='coral', linewidth=2, linestyle='--', label='Cross-entropy H(p, q=0.6)')
axes[1].plot(p, -(p * np.log(0.9) + (1-p) * np.log(0.1)),
             color='tomato', linewidth=2, linestyle=':', label='Cross-entropy H(p, q=0.9)')
axes[1].set_title("Cross-Entropy vs Entropy\nGap = how wrong your model is")
axes[1].set_xlabel("True probability p")
axes[1].set_ylabel("Loss")
axes[1].set_ylim(0, 3)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Loss 1 · Binary Cross-Entropy (BCE)

**Use when:** Exactly **2 classes** — spam/not spam, disease/healthy, fraud/legit.  
Your output layer has **1 neuron**. After sigmoid → probability between 0 and 1.

$$\mathcal{L}_{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \Big[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \Big]$$

**Reading the formula:**  
- When $y=1$: only the left term matters → penalise if $\hat{y}$ is small  
- When $y=0$: only the right term matters → penalise if $\hat{y}$ is large  

The two terms together ensure **both directions** of wrong prediction are punished.

**PyTorch versions:**

| Function | Input | Notes |
|---|---|---|
| `nn.BCELoss` | Probabilities (after sigmoid) | You apply sigmoid manually |
| `nn.BCEWithLogitsLoss` | Raw logits | Sigmoid done internally ✓ preferred |

## BCE FROM SCRATCH

In [ ]:
y_true = torch.tensor([1.0,0.0,1.0,0.0,1.0])
logits= torch.tensor([2.0,-1.5,0.3,1.8,-0.5])
y_pred=torch.sigmoid(logits)
print("Logits      :", logits.numpy().round(2))
print("Probs       :", y_pred.detach().numpy().round(3))
print("Ground truth:", y_true.numpy())

In [ ]:
### BCE from scratch
eps = 1e-8
bce_scratch = -(
    y_true * torch.log(y_pred + eps) +
    (1 - y_true) * torch.log(1 - y_pred + eps)
).mean()

### BCE Built-in in PyTorch
bce_builtin = nn.BCEWithLogitsLoss()(logits, y_true)

print(f"\nBCE (scratch) : {bce_scratch.item():.6f}")
print(f"BCE (PyTorch) : {bce_builtin.item():.6f}")

per_sample = -(
    y_true * torch.log(y_pred + eps) +
    (1 - y_true) * torch.log(1 - y_pred + eps)
)

print("\nPer-sample loss:", per_sample.detach().numpy().round(4))
print("(higher = model was more wrong on that sample)")


### Visualizing BCE Loss

In [ ]:
p = np.linspace(0.001, 0.999, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Binary Cross-Entropy Deep Dive", fontsize=13, fontweight='bold')

# Left: both terms
axes[0].plot(p, -np.log(p),       color='steelblue', linewidth=2.5, label='-log(p)   when y=1')
axes[0].plot(p, -np.log(1-p),     color='coral',     linewidth=2.5, label='-log(1-p) when y=0')
axes[0].set_title("Two Terms of BCE")
axes[0].set_xlabel("Predicted probability")
axes[0].set_ylabel("Loss")
axes[0].set_ylim(0, 5)
axes[0].legend(); axes[0].grid(alpha=0.3)

scenarios = [0.05, 0.2, 0.5, 0.8, 0.95]
x_pos = np.arange(len(scenarios))
losses_y1 = [-np.log(p) for p in scenarios]
losses_y0 = [-np.log(1-p) for p in scenarios]

w = 0.35
axes[1].bar(x_pos - w/2, losses_y1, w, color='steelblue', alpha=0.8, label='y=1 (want high prob)')
axes[1].bar(x_pos + w/2, losses_y0, w, color='coral',     alpha=0.8, label='y=0 (want low prob)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'p={s}' for s in scenarios])
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss at Different Confidence Levels")
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')
axes[1].set_ylim(0, 6)

# Right: total BCE loss surface
p_grid = np.linspace(0.01, 0.99, 300)
y1_loss = -np.log(p_grid)
y0_loss = -np.log(1 - p_grid)
total = (y1_loss + y0_loss) / 2
axes[2].plot(p_grid, y1_loss,  color='steelblue', linewidth=2, alpha=0.6, label='y=1 term')
axes[2].plot(p_grid, y0_loss,  color='coral',     linewidth=2, alpha=0.6, label='y=0 term')
axes[2].plot(p_grid, total,    color='black',      linewidth=2.5, label='Average BCE')
axes[2].set_title("Combined BCE Surface")
axes[2].set_xlabel("Predicted probability")
axes[2].set_ylabel("Loss")
axes[2].set_ylim(0, 5)
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Loss 2 · Cross-Entropy (Multi-Class)

**Use when:** 3 or more **mutually exclusive** classes — cat/dog/bird, digits 0–9.  
Your output layer has **one neuron per class**.

### The pipeline:

$$\text{logits} \xrightarrow{\text{softmax}} \text{probabilities} \xrightarrow{-\log} \text{loss}$$

### Softmax

Converts raw logits to a valid probability distribution (sums to 1):

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

### Cross-Entropy

Once we have probabilities, we only care about the **true class probability**:

$$\mathcal{L}_{CE} = -\frac{1}{N} \sum_{i=1}^{N} \log(\hat{p}_{i,\, c_i})$$

where $c_i$ is the true class index for sample $i$.

**PyTorch's `nn.CrossEntropyLoss` does softmax + log + NLL in one step.**  
Never apply softmax manually before passing logits to it.

In [ ]:
logits = torch.tensor([3.0, 1.0, 0.5, -1.0])

# --- Naive softmax (unstable for large values) ---
def softmax_naive(z):
    e = torch.exp(z)
    return e / e.sum()

# --- Stable softmax (subtract max first) ---
def softmax_stable(z):
    e = torch.exp(z - z.max())   # subtracting max doesn't change output
    return e / e.sum()

naive  = softmax_naive(logits)
stable = softmax_stable(logits)
pytorch = F.softmax(logits, dim=0)

print("Logits          :", logits.numpy())
print("Softmax (naive) :", naive.numpy().round(4))
print("Softmax (stable):", stable.numpy().round(4))
print("Softmax (PyTorch):", pytorch.numpy().round(4))
print("Sum (must = 1.0):", stable.sum().item())

# --- Show why stability matters ---
big_logits = torch.tensor([1000.0, 999.0, 998.0])
print("\nWith large logits:")
try:
    print("Naive :", softmax_naive(big_logits))    # will be nan/inf
except:
    print("Naive : OVERFLOW!")
print("Stable:", softmax_stable(big_logits).numpy().round(4))

### Cross-Entropy From Scratch

In [ ]:
# 5 samples, 4 classes (e.g. cat/dog/bird/fish)
logits = torch.tensor([
    [3.0,  1.0,  0.5, -1.0],   # confident → cat
    [0.2,  2.8,  0.1,  0.3],   # confident → dog
    [-0.5, 0.3,  2.5,  0.1],   # confident → bird
    [1.0,  1.1,  0.9,  1.0],   # uncertain (all similar)
    [0.1,  0.2,  0.1,  3.0],   # confident → fish
])

y_true = torch.tensor([0, 1, 2, 1, 3])   # true class indices

# Step 1: softmax
def softmax_batch(z):
    e = torch.exp(z - z.max(dim=1, keepdim=True).values)
    return e / e.sum(dim=1, keepdim=True)

probs = softmax_batch(logits)

# Step 2: pick probability of true class per sample
N = logits.shape[0]
true_class_probs = probs[range(N), y_true]

# Step 3: negative log mean
ce_scratch = -torch.log(true_class_probs).mean()
ce_pytorch = nn.CrossEntropyLoss()(logits, y_true)

print("Probabilities (softmax output):")
print(probs.detach().numpy().round(3))
print("\nTrue class probabilities:", true_class_probs.detach().numpy().round(3))
print(f"\nCE (scratch) : {ce_scratch.item():.5f}")
print(f"CE (PyTorch) : {ce_pytorch.item():.5f}")

# Per-sample loss
per_sample_loss = -torch.log(true_class_probs)
print("\nPer-sample loss:", per_sample_loss.detach().numpy().round(4))
print("(sample 3 = uncertain → highest loss)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Cross-Entropy Deep Dive", fontsize=13, fontweight='bold')

# --- Left: softmax output for each sample ---
sample_labels = ['Sample 1\n(cat)', 'Sample 2\n(dog)', 'Sample 3\n(bird)',
                 'Sample 4\n(uncertain)', 'Sample 5\n(fish)']
class_names = ['Cat', 'Dog', 'Bird', 'Fish']
colors_bar = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
probs_np = probs.detach().numpy()

x = np.arange(len(class_names))
width = 0.15
for i, (label, row) in enumerate(zip(sample_labels, probs_np)):
    axes[0].bar(x + i*width, row, width, label=label, alpha=0.8)
axes[0].set_xticks(x + width*2)
axes[0].set_xticklabels(class_names)
axes[0].set_ylabel("Probability")
axes[0].set_title("Softmax Output Per Sample")
axes[0].legend(fontsize=7, loc='upper right')
axes[0].grid(alpha=0.3, axis='y')

# Middle: per-sample loss
losses_np = per_sample_loss.detach().numpy()
bar_colors = ['green' if l < 0.5 else 'orange' if l < 1.5 else 'red' for l in losses_np]
axes[1].bar(range(N), losses_np, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(range(N))
axes[1].set_xticklabels([f'S{i+1}' for i in range(N)])
axes[1].set_ylabel("Loss")
axes[1].set_title("Per-Sample CE Loss\nGreen=good, Orange=ok, Red=bad")
axes[1].axhline(ce_pytorch.item(), color='black', linestyle='--', label=f'Mean={ce_pytorch.item():.3f}')
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

# Right: loss vs probability of true class
p_range = np.linspace(0.001, 0.999, 300)
axes[2].plot(p_range, -np.log(p_range), color='steelblue', linewidth=2.5)
axes[2].fill_between(p_range, -np.log(p_range), alpha=0.1, color='steelblue')
for i, (prob, loss) in enumerate(zip(true_class_probs.detach(), losses_np)):
    axes[2].scatter(prob.item(), loss, s=100, zorder=5, label=f'S{i+1}: p={prob:.2f}')
axes[2].set_xlabel("P(true class)")
axes[2].set_ylabel("-log(p)")
axes[2].set_title("Each Sample on the Loss Curve")
axes[2].set_ylim(0, 5); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Loss 3 · Categorical Cross-Entropy vs Cross-Entropy

People often use these terms interchangeably, but in PyTorch there's
a practical difference — the **format of the labels**.

| | Labels format | PyTorch function |
|---|---|---|
| **Cross-Entropy** | Class indices `[0, 2, 1]` | `nn.CrossEntropyLoss` |
| **Categorical CE** | One-hot vectors `[[1,0,0], [0,0,1]]` | Same loss, convert first |

The math is **identical** — just different ways to express "which class is correct."

One-hot encoding:
$$[0, 1, 2] \rightarrow \begin{bmatrix}1&0&0\\0&1&0\\0&0&1\end{bmatrix}$$

In [ ]:
logits = torch.tensor([
    [2.0, 0.5, 0.1],
    [0.1, 0.2, 3.0],
    [0.5, 2.5, 0.3],
])

# Format 1: class indices
y_indices = torch.tensor([0, 2, 1])

# Format 2: one-hot
y_onehot = F.one_hot(y_indices, num_classes=3).float()

print("Class indices:", y_indices.numpy())
print("One-hot:\n", y_onehot.numpy())

# --- CrossEntropyLoss: expects class indices ---
ce_indices = nn.CrossEntropyLoss()(logits, y_indices)

# --- Categorical CE from scratch with one-hot ---
log_probs = F.log_softmax(logits, dim=1)        # log(softmax(z))
ce_onehot = -(y_onehot * log_probs).sum(dim=1).mean()

print(f"\nCE (indices, PyTorch) : {ce_indices.item():.6f}")
print(f"CE (one-hot, scratch) : {ce_onehot.item():.6f}")
print("Identical? :", torch.allclose(
    torch.tensor(ce_indices.item()),
    torch.tensor(ce_onehot.item()), atol=1e-5
))

## Loss 4 · Negative Log-Likelihood (NLL)

`nn.NLLLoss` is the final piece of the Cross-Entropy puzzle.

$$\mathcal{L}_{NLL} = -\frac{1}{N} \sum_i \log p_{i, c_i}$$

The relationship between them:

$$\text{CrossEntropyLoss}(\text{logits}) = \text{NLLLoss}(\text{log\_softmax}(\text{logits}))$$

In other words — **CrossEntropyLoss = LogSoftmax + NLLLoss** combined.

**When would you use NLLLoss directly?**  
When your model already outputs log-probabilities (e.g. models with
`log_softmax` as the final activation). You avoid doing softmax twice.

In [ ]:
logits = torch.tensor([
    [2.0, 0.5, 0.1],
    [0.1, 0.2, 3.0],
    [0.5, 2.5, 0.3],
])
y_true = torch.tensor([0, 2, 1])

# CrossEntropyLoss in one shot
ce = nn.CrossEntropyLoss()(logits, y_true)

# The same thing manually: LogSoftmax → NLLLoss
log_probs = F.log_softmax(logits, dim=1)    # log(softmax(z))
nll = nn.NLLLoss()(log_probs, y_true)

print(f"CrossEntropyLoss       : {ce.item():.6f}")
print(f"LogSoftmax + NLLLoss   : {nll.item():.6f}")
print("Same?", torch.allclose(ce, nll))

print("\nlog_softmax output (what NLLLoss receives):")
print(log_probs.detach().numpy().round(4))
print("(these are log-probabilities — all negative, closer to 0 = more confident)")

## Loss 5 · Label Smoothing Cross-Entropy

**Problem with standard CE:** The model can become **overconfident** —  
it pushes probability of true class toward 1.0 and everything else to 0.  
This makes it fragile on unseen data.

**Label Smoothing** softens the hard one-hot targets:

$$y_{smooth} = (1 - \epsilon) \cdot y_{one\_hot} + \frac{\epsilon}{K}$$

Instead of `[0, 1, 0, 0]` you train toward `[0.025, 0.925, 0.025, 0.025]`  
(with $\epsilon=0.1$, $K=4$ classes)

**Effect:** The model is never "perfectly confident" — improves generalisation.  
Used in almost all modern vision and NLP models.

PyTorch has this built in: `nn.CrossEntropyLoss(label_smoothing=0.1)`

In [ ]:
logits = torch.tensor([
    [4.0, 0.5, 0.1, 0.1],   # very confident → class 0
    [0.1, 0.2, 4.0, 0.1],   # very confident → class 2
])
y_true = torch.tensor([0, 2])

# Standard CE
ce_standard = nn.CrossEntropyLoss()(logits, y_true)

# Label Smoothing CE
ce_smooth_01 = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, y_true)
ce_smooth_02 = nn.CrossEntropyLoss(label_smoothing=0.2)(logits, y_true)

print(f"CE (no smoothing)       : {ce_standard.item():.4f}")
print(f"CE (smoothing=0.1)      :, {ce_smooth_01.item():.4f}")
print(f"CE (smoothing=0.2)      :, {ce_smooth_02.item():.4f}")

# Show what the smoothed labels look like
epsilon = 0.1
K = 4
y_onehot = F.one_hot(y_true, num_classes=K).float()
y_smooth  = (1 - epsilon) * y_onehot + epsilon / K
print("\nHard labels:\n",    y_onehot.numpy())
print("Smoothed labels:\n", y_smooth.numpy())

## Loss 6 · Focal Loss (Class Imbalance)

**Problem:** In many real datasets, classes are heavily imbalanced.  
Example — fraud detection: 99% legitimate, 1% fraud.

With standard CE, the model just learns to predict "not fraud" for everything  
and still achieves 99% accuracy. The rare class gets ignored.

**Focal Loss** adds a modulating factor $(1-p_t)^\gamma$ that:
- Down-weights easy examples (things the model already predicts correctly)
- Up-weights hard examples (the rare minority class)

$$\mathcal{L}_{focal} = -\frac{1}{N} \sum_i (1 - \hat{p}_{i,c_i})^\gamma \cdot \log(\hat{p}_{i,c_i})$$

- $\gamma = 0$ → standard Cross-Entropy  
- $\gamma = 2$ → most common setting  
- Higher $\gamma$ → more focus on hard examples

In [ ]:
def focal_loss(logits, y_true, gamma=2.0):
    ce_loss = F.cross_entropy(logits, y_true, reduction='none')
    probs   = F.softmax(logits, dim=1)
    N       = logits.shape[0]
    pt      = probs[range(N), y_true]             # prob of true class
    weight  = (1 - pt) ** gamma                   # modulating factor
    return (weight * ce_loss).mean()

# Simulate imbalanced scenario:
# Easy samples (model is already confident) vs hard samples
logits_easy = torch.tensor([[4.0, 0.1, 0.1]] * 4)   # model very confident
logits_hard = torch.tensor([[0.4, 0.3, 0.3]] * 4)   # model uncertain
y = torch.tensor([0, 0, 0, 0])

print("Easy samples (model already confident)")
print(f"CE Loss    : {F.cross_entropy(logits_easy, y).item():.4f}")
print(f"Focal γ=2  : {focal_loss(logits_easy, y, gamma=2).item():.4f}")
print(f"Focal down-weighs easy samples smaller loss")

print("\nHard samples (model uncertain)")
print(f"CE Loss    : {F.cross_entropy(logits_hard, y).item():.4f}")
print(f"Focal γ=2  : {focal_loss(logits_hard, y, gamma=2).item():.4f}")
print(f"Focal keeps hard samples weighted similarly")

In [ ]:
p = np.linspace(0.01, 0.99, 300)
ce = -np.log(p)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Focal Loss vs Cross-Entropy", fontsize=13, fontweight='bold')

#Left: loss curves for different gamma
gammas = [0, 0.5, 1, 2, 5]
colors = ['black', 'steelblue', 'dodgerblue', 'coral', 'red']
for gamma, color in zip(gammas, colors):
    focal = (1 - p) ** gamma * (-np.log(p))
    label = f'γ={gamma}' + (' (= CE)' if gamma == 0 else '')
    axes[0].plot(p, focal, color=color, linewidth=2, label=label)
axes[0].set_xlabel("Probability of true class p_t")
axes[0].set_ylabel("Loss")
axes[0].set_title("Focal Loss for Different γ\nHigher γ = more focus on hard examples")
axes[0].set_ylim(0, 4); axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: relative weight given to easy vs hard
easy_p = np.array([0.9, 0.8, 0.7, 0.5, 0.3, 0.1])
weights_g0 = (1 - easy_p) ** 0
weights_g2 = (1 - easy_p) ** 2
weights_g5 = (1 - easy_p) ** 5

x_pos = np.arange(len(easy_p))
w = 0.25
axes[1].bar(x_pos - w, weights_g0, w, label='γ=0 (CE)', color='black',     alpha=0.7)
axes[1].bar(x_pos,     weights_g2, w, label='γ=2',      color='coral',      alpha=0.7)
axes[1].bar(x_pos + w, weights_g5, w, label='γ=5',      color='red',        alpha=0.7)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'p={p}' for p in easy_p])
axes[1].set_ylabel("Modulating weight (1-p)^γ")
axes[1].set_title("How Much Each Sample is Weighted\n(p=0.9 easy, p=0.1 hard)")
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Same 3 samples, compare what each loss gives
logits = torch.tensor([
    [2.0, 0.3, 0.1],   # fairly confident, correct (class 0)
    [0.8, 0.9, 0.7],   # uncertain
    [0.1, 0.2, 3.5],   # very confident, correct (class 2)
])
y = torch.tensor([0, 1, 2])

ce      = nn.CrossEntropyLoss(reduction='none')(logits, y)
ce_sm   = nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)(logits, y)
focal_2 = torch.tensor([focal_loss(logits[i:i+1], y[i:i+1], gamma=2).item() for i in range(3)])

print(f"{'Sample':<10} {'True':>6} {'CE':>8} {'CE+smooth':>12} {'Focal γ=2':>12}")
print("-" * 52)
for i in range(3):
    probs = F.softmax(logits[i], dim=0)
    print(f"Sample {i+1:<3}  class={y[i].item()}  {ce[i].item():>8.4f}  {ce_sm[i].item():>12.4f}  {focal_2[i].item():>12.4f}")

## Final Summary — Which Loss to Use

| Scenario | Loss Function | Key Point |
|---|---|---|
| Binary (2 classes) | `nn.BCEWithLogitsLoss` | 1 output neuron, no sigmoid before loss |
| Multi-class (3+) | `nn.CrossEntropyLoss` | N output neurons, no softmax before loss |
| Multi-label (multiple true classes) | `nn.BCEWithLogitsLoss` | Treat each class independently |
| Overconfident model | `CrossEntropyLoss(label_smoothing=0.1)` | Soft targets instead of hard 0/1 |
| Class imbalance | Focal Loss | Down-weight easy, up-weight hard |
| Model outputs log-probs | `nn.NLLLoss` | Skip the softmax step |

**The golden rule:**  
→ Never apply sigmoid/softmax inside your model when using PyTorch loss functions.  
→ Always pass **raw logits** to the loss — it handles the activation internally and is numerically more stable.

**Next up → M4: Backpropagation — now that we have a loss, let's learn how to reduce it.**